# 03 - Feature Engineering

Builds every feature used by the model, and splits the data into train/validation/test.
All the logic lives in `src/features.py` and `src/graph_features.py` -- this notebook calls
it and checks the result.

## Data leakage, with a concrete example

Data leakage is when a feature accidentally contains information that wouldn't actually
exist yet at the moment a real prediction is made -- usually something from the future.

**Example with this data:** say we're building a feature for account `10_8123FB9B0`'s
transaction on Sept 3rd -- "average amount this account usually sends." If that average is
computed using ALL of the account's transactions, including ones from Sept 10th or 15th,
we've used information that didn't exist yet on Sept 3rd. A real bank cannot know on Sept 3rd
what an account will do on Sept 15th. A model trained this way looks great in testing, then
gets quietly worse in the real world, where that future information simply isn't there.

This is also why the train/val/test split is done by TIME, not randomly: a randomly-placed
training row from Sept 15th could indirectly teach the model something about a test row from
Sept 3rd (same account, similar behaviour) -- leaking the future into the past again.

**One wrinkle specific to this dataset:** timestamps only have minute precision, and a LOT of
transactions share the exact same minute. So "strictly before" has to mean strictly before the
timestamp VALUE, not just earlier in row order -- two transactions in the same minute must
never be allowed to use each other as "the past", since we can't know which came first.

The **No future data check** section near the end of this notebook verifies this directly.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

from src.config import load_config
from src.features import build_features, time_based_split, add_transaction_features, add_account_behavior_features
from src.graph_features import add_graph_features, save_outputs

config = load_config(project_root / "config.yaml")

## 1. Time-based split

Sort by timestamp, then cut by row position: train gets the earliest rows, val the next
chunk, test the most recent rows. Ratios come from `config.yaml` (`split.train_frac` etc).

In [ ]:
df = pd.read_parquet(config["paths"]["clean_file"]).sort_values("timestamp").reset_index(drop=True)
train, val, test = time_based_split(df, config["split"])

for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: {len(split):,} rows, {split['timestamp'].min()} to {split['timestamp'].max()}")

assert train["timestamp"].max() <= val["timestamp"].min(), "train/val overlap in time!"
assert val["timestamp"].max() <= test["timestamp"].min(), "val/test overlap in time!"
print("\nNo time overlap between splits: confirmed")

## 2. Transaction features

Only use each row's own fields, so there's no leakage risk: amount in USD, log-amount
(handles the huge range of amounts -- see Phase 1/2), hour, day of week, a cross-currency
flag, and a cross-bank flag.

In [ ]:
# NOTE: features are computed on the FULL chronologically-sorted dataset, not per-split,
# and split AFTER. That looks backwards but is correct -- see src/features.py's docstring:
# a val-set account genuinely does have real history from the train period, and a real
# deployed system doesn't forget an account's past at an arbitrary split boundary. The
# "no future data" rule is enforced per-row (strictly before THAT row's own timestamp), which
# holds regardless of which split a row ends up in.
df_feat = add_transaction_features(df)
df_feat[["timestamp", "amount_usd", "log_amount", "hour", "day_of_week", "is_cross_currency", "is_cross_bank"]].head()

## 3. Account behaviour features

The leakage-sensitive part. Every one of these uses ONLY an account's transactions strictly
before the current row's timestamp:

- `sent_count_1d` / `sent_count_7d`: how many transactions this account sent in the trailing
  1 / 7 days
- `sent_avg_amount_7d` / `sent_max_amount_7d`: average / largest amount sent recently
- `unique_counterparties_7d`: how many different accounts it sent to recently (a sudden jump
  here looks like fan-out behaviour)
- `sent_avg_amount_alltime` / `amount_vs_usual_ratio`: this transaction's amount compared to
  what's "normal" for the account -- a transaction far above its own usual amount stands out
- `minutes_since_prev_txn`: time since the account's last transaction
- `received_count_1d` / `received_count_7d`: same idea, but counting money coming IN

In [ ]:
df_feat = add_account_behavior_features(df_feat)

cols = ["timestamp", "from_id", "amount_usd", "sent_count_1d", "sent_count_7d",
        "sent_avg_amount_7d", "unique_counterparties_7d", "amount_vs_usual_ratio",
        "minutes_since_prev_txn"]

# Look at one account with several transactions, to see the features build up over time
busiest_account = df_feat["from_id"].value_counts().idxmax()
df_feat[df_feat["from_id"] == busiest_account][cols].head(10)

## 4. Graph features

Built with `networkx` in `src/graph_features.py`. Money laundering often shows up in the
SHAPE of an account's connections rather than in any single transaction:

- `in_degree` / `out_degree`: how many different accounts sent to / received from this
  account (fan-in / fan-out) -- a layering account moving money through many shells at once
  often has an unusually high degree.
- `total_in_amount` / `total_out_amount`: total money received / sent. An account passing
  through roughly what it takes in (in ~ out) looks like a pass-through / layering account,
  not one that's genuinely spending or saving.
- `pagerank`: how "central" an account is in the money-flow network, weighted by amount --
  scored higher when money flows in from OTHER already-central accounts. Useful for spotting
  hub accounts in the middle of a network, not just ones with a high raw transaction count.

Leakage rule here is coarser than the per-row rule above: each split's graph is built from
edges up to the END of that split's own time range (train graph = train edges only, val
graph = train+val edges, test graph = everything). Test-period data never reaches train or
val features. Both the sender's (`from_`) and receiver's (`to_`) graph position are attached
to each transaction.

In [ ]:
train, val, test = build_features(config)
train, val, test = add_graph_features(train, val, test, config["features"]["pagerank_alpha"])

graph_cols = ["from_in_degree", "from_out_degree", "from_total_in_amount", "from_pagerank",
              "to_in_degree", "to_out_degree", "to_pagerank"]
train[["timestamp", "from_id", "to_id"] + graph_cols].head()

## 5. Save feature tables

In [ ]:
save_outputs(train, val, test, config)
print(f"train: {train.shape}")
print(f"val:   {val.shape}")
print(f"test:  {test.shape}")

## No future data check

A direct test, not just a claim: for a random sample of rows, independently recompute
`sent_count_7d` with a plain, obviously-correct (if slow) loop -- strictly-before comparison
on the raw timestamps, sharing no code with `src/features.py` -- and confirm it matches
exactly. Then, separately, corrupt every amount in the second half of the dataset (the
"future" relative to the first half) and confirm that NONE of the first half's features
change as a result.

In [ ]:
def brute_force_sent_count_7d(data, row_idx):
    row = data.loc[row_idx]
    window_start = row["timestamp"] - pd.Timedelta("7D")
    mask = (
        (data["from_id"] == row["from_id"])
        & (data["timestamp"] >= window_start)  # pandas closed="left" includes the exact boundary
        & (data["timestamp"] < row["timestamp"])  # STRICTLY before -- the actual leakage check
    )
    return mask.sum()


rng = np.random.default_rng(config["project"]["random_seed"])
sample_idx = rng.choice(df_feat.index, size=min(300, len(df_feat)), replace=False)

mismatches = 0
for idx in sample_idx:
    expected = brute_force_sent_count_7d(df_feat, idx)
    actual = df_feat.loc[idx, "sent_count_7d"]
    if expected != actual:
        mismatches += 1
        print(f"MISMATCH at row {idx}: expected {expected}, got {actual}")

print(f"Checked {len(sample_idx)} random rows against an independent brute-force count.")
print(f"Mismatches: {mismatches} (must be 0)")
assert mismatches == 0

In [ ]:
# Corrupt every transaction in the FUTURE half of the dataset, then confirm the PAST
# half's account-behaviour features are completely unchanged.
midpoint = df_feat["timestamp"].median()
df_corrupted = df.copy()
future_mask = df_corrupted["timestamp"] > midpoint
df_corrupted.loc[future_mask, "amount_usd"] = df_corrupted.loc[future_mask, "amount_usd"] + 1_000_000_000

df_corrupted_feat = add_account_behavior_features(add_transaction_features(df_corrupted))

past_mask = df_feat["timestamp"] <= midpoint
check_cols = ["sent_count_7d", "sent_avg_amount_7d", "sent_avg_amount_alltime", "unique_counterparties_7d"]

changed = 0
for col in check_cols:
    original = df_feat.loc[past_mask, col].fillna(-999)
    corrupted = df_corrupted_feat.loc[past_mask, col].fillna(-999)
    n_changed = (~np.isclose(original, corrupted)).sum()
    changed += n_changed
    print(f"{col}: {n_changed} past rows changed after corrupting the future (must be 0)")

assert changed == 0
print("\nNo future data reaches past rows: confirmed")